# 05 MobileNetV2 Transfer Learning

## Objective
Train a MobileNetV2 model with transfer learning:
- ✓ Load pretrained MobileNetV2
- ✓ Freeze base layers
- ✓ Add custom top layers
- ✓ Train with frozen base (phase 1)
- ✓ Unfreeze and fine-tune (phase 2)
- ✓ Evaluate and compare with CNN baseline
- ✓ Save best model

**Prerequisites:** `01_data_check.ipynb`, `02_dataset_visualization.ipynb`, `03_data_preprocessing.ipynb`, `04_cnn_baseline.ipynb`

### Import Libraries

In [10]:
import sys
import os

sys.path.insert(0, '../utils')

from model_utils import (
    build_mobilenet_model,
    unfreeze_model_layers,
    train_model,
    evaluate_model,
    save_model,
    save_metrics,
    get_model_summary,
    compute_class_weights_from_labels
)
from visualization_utils import (
    plot_training_history,
    plot_confusion_matrix,
    plot_metrics_comparison
)

import numpy as np
import tensorflow as tf
from tensorflow import keras
import json
import matplotlib.pyplot as plt

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

✓ Libraries imported successfully!
TensorFlow version: 2.15.0


### Load Preprocessed Data

In [11]:
print("Loading preprocessed data...\n")

# Load data
X_train = np.load('../results/preprocessed_data/X_train.npy')
y_train = np.load('../results/preprocessed_data/y_train.npy')
X_val = np.load('../results/preprocessed_data/X_val.npy')
y_val = np.load('../results/preprocessed_data/y_val.npy')
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)

print(f"✓ Data loaded successfully")
print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"\nNumber of classes: {num_classes}")

Loading preprocessed data...

✓ Data loaded successfully

Data shapes:
  X_train: (532, 224, 224, 3)
  X_val: (114, 224, 224, 3)
  X_test: (115, 224, 224, 3)

Number of classes: 40


In [12]:
class_names = [label for label, _ in sorted(label_encoding.items(), key=lambda item: item[1])]
class_weight = compute_class_weights_from_labels(y_train, class_names=class_names, verbose=True)



Class weights computed from y_train
(Using class weights instead of SMOTE keeps the original image distribution intact.)
----------------------------------------------------------------------
   0 | Brinjal_Healthy                     | samples:    41 | weight: 0.3244
   1 | Brinjal_Cercospora                  | samples:    12 | weight: 1.1083
   2 | Brinjal_Early_Leaf_spot             | samples:    12 | weight: 1.1083
   3 | Brinjal_Leaf_curl                   | samples:     9 | weight: 1.4778
   4 | Brinjal_Little_leaf                 | samples:    10 | weight: 1.3300
   5 | Brinjal_Magnessium                  | samples:    10 | weight: 1.3300
   6 | Brinjal_Mosaic                      | samples:     6 | weight: 2.2167
   7 | Brinjal_Nitrogen                    | samples:    10 | weight: 1.3300
   8 | Brinjal_pest_damage                 | samples:     8 | weight: 1.6625
   9 | Brinjal_Pottasium                   | samples:    23 | weight: 0.5783
  10 | Castor_Healthy                

In [13]:
# Quick MobileNetV2 smoke test: short train to verify transfer learning helps
TEST_EPOCHS_TL = 8
BATCH_SIZE_TL = 32
print("Building MobileNetV2 (for short test) and running a short training pass...")
input_shape = (224, 224, 3)
model_tl = build_mobilenet_model(num_classes=num_classes, input_shape=input_shape, trainable_layers=0)
print(get_model_summary(model_tl))
# Simple callbacks (reuse pattern from baseline)
callbacks_tl = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint('../models/mobilenet/best_model_tl.h5', save_best_only=True, monitor='val_accuracy', verbose=0)
]
import os
os.makedirs('../models/mobilenet', exist_ok=True)
history_tl = train_model(model_tl, (X_train, y_train), (X_val, y_val), epochs=TEST_EPOCHS_TL, batch_size=BATCH_SIZE_TL, callbacks=callbacks_tl, class_weight=class_weight)
print("Short transfer-learning training complete.")
metrics_tl = evaluate_model(model_tl, X_test, y_test, class_names)
print(metrics_tl.keys())


Building MobileNetV2 (for short test) and running a short training pass...
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling_2 (Rescaling)     (None, 224, 224, 3)       0         
                                                                 
 mobilenetv2_preprocessing   (None, 224, 224, 3)       0         
 (Rescaling)                                                     
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d_2  (None, 1280)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dense_4 (Dense)             (None, 128)     

c:\Users\GAME-PC\anaconda3\envs\cnn_model\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


17/17 [==============================] - 6s 259ms/step - loss: 3.8948 - accuracy: 0.0338 - val_loss: 3.9810 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 2/8
17/17 [==============================] - 4s 218ms/step - loss: 3.9148 - accuracy: 0.0263 - val_loss: 3.9236 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 3/8
17/17 [==============================] - 4s 215ms/step - loss: 3.8637 - accuracy: 0.0338 - val_loss: 3.8779 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 4/8
17/17 [==============================] - 4s 214ms/step - loss: 3.8324 - accuracy: 0.0338 - val_loss: 3.8523 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 5/8
17/17 [==============================] - 4s 216ms/step - loss: 3.8128 - accuracy: 0.0301 - val_loss: 3.8351 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 6/8
17/17 [==============================] - 4s 215ms/step - loss: 3.7921 - accuracy: 0.0244 - val_loss: 3.8159 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 7/8
17/17 [==============================] - 4s 214ms/st

### Configuration

In [ ]:
# Training configuration
EPOCHS_PHASE1 = 40
EPOCHS_PHASE2 = 25
BATCH_SIZE = 16
TRAINABLE_LAYERS = 80
LEARNING_RATE_PHASE1 = 0.001
LEARNING_RATE_PHASE2 = 0.00001
DENSE_UNITS = 256
DROPOUT_RATE = 0.35
MODEL_NAME = 'mobilenet_v2'

print("Training Configuration:")
print(f"  Model: MobileNetV2 (Transfer Learning)")
print(f"  Phase 1 Epochs: {EPOCHS_PHASE1} (frozen base)")
print(f"  Phase 2 Epochs: {EPOCHS_PHASE2} (fine-tuned)")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Trainable layers (phase 2): {TRAINABLE_LAYERS}")
print(f"  Phase 1 LR: {LEARNING_RATE_PHASE1}")
print(f"  Phase 2 LR: {LEARNING_RATE_PHASE2}")
print(f"  Dense units: {DENSE_UNITS}")
print(f"  Dropout rate: {DROPOUT_RATE}")

Training Configuration:
  Model: MobileNetV2 (Transfer Learning)
  Phase 1 Epochs: 30 (frozen base)
  Phase 2 Epochs: 20 (fine-tuned)
  Batch Size: 32
  Trainable layers (phase 2): 50


### 1. Build MobileNetV2 Model

In [ ]:
print("\nBuilding MobileNetV2 model...\n")

input_shape = (224, 224, 3)
model = build_mobilenet_model(
    num_classes=num_classes,
    input_shape=input_shape,
    trainable_layers=0,  # Start with frozen base
    learning_rate=LEARNING_RATE_PHASE1,
    optimizer='adam',
    dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE
)

print("✓ Model built successfully!\n")
print("Model Summary:")
print(get_model_summary(model))


Building MobileNetV2 model...

✓ Model built successfully!

Model Summary:
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling_3 (Rescaling)     (None, 224, 224, 3)       0         
                                                                 
 mobilenetv2_preprocessing   (None, 224, 224, 3)       0         
 (Rescaling)                                                     
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d_3  (None, 1280)              0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dense_6 (Dense)             (None, 128)    

### 2. Setup Callbacks for Phase 1

In [16]:
os.makedirs('../models/mobilenet', exist_ok=True)

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

print("✓ Phase 1 callbacks configured (frozen base)")

✓ Phase 1 callbacks configured (frozen base)


### 3. Phase 1: Train with Frozen Base

In [17]:
print("\n" + "="*70)
print("PHASE 1: Training with Frozen Base Layers")
print("="*70 + "\n")

history_phase1 = train_model(
    model=model,
    train_data=(X_train, y_train),
    val_data=(X_val, y_val),
    epochs=EPOCHS_PHASE1,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase1,
    class_weight=class_weight
)

print("\n✓ Phase 1 training completed!")


PHASE 1: Training with Frozen Base Layers

Epoch 1/30
17/17 [==============================] - 5s 243ms/step - loss: 3.9676 - accuracy: 0.0320 - val_loss: 4.0064 - val_accuracy: 0.0175 - lr: 1.0000e-04
Epoch 2/30
17/17 [==============================] - 4s 217ms/step - loss: 3.8626 - accuracy: 0.0188 - val_loss: 3.9437 - val_accuracy: 0.0263 - lr: 1.0000e-04
Epoch 3/30
17/17 [==============================] - 4s 217ms/step - loss: 3.8845 - accuracy: 0.0132 - val_loss: 3.9016 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 4/30
17/17 [==============================] - 4s 220ms/step - loss: 3.8178 - accuracy: 0.0263 - val_loss: 3.8725 - val_accuracy: 0.0351 - lr: 1.0000e-04
Epoch 5/30
17/17 [==============================] - 4s 216ms/step - loss: 3.8723 - accuracy: 0.0150 - val_loss: 3.8461 - val_accuracy: 0.0439 - lr: 1.0000e-04
Epoch 6/30
17/17 [==============================] - 4s 216ms/step - loss: 3.8049 - accuracy: 0.0395 - val_loss: 3.8289 - val_accuracy: 0.0351 - lr: 1.0000e-04
Ep

### 4. Phase 2: Fine-tune with Unfrozen Layers

print("\n" + "="*70)
print("PHASE 2: Fine-tuning with Unfrozen Layers")
print("="*70 + "\n")

# Unfreeze layers
model = unfreeze_model_layers(model, num_layers=TRAINABLE_LAYERS, learning_rate=LEARNING_RATE_PHASE2, optimizer='adam')

print(f"✓ Unfroze last {TRAINABLE_LAYERS} layers")
print(f"  Learning rate reduced to {LEARNING_RATE_PHASE2}\n")

# Callbacks for phase 2
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-8,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        '../models/mobilenet/best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

history_phase2 = train_model(
    model=model,
    train_data=(X_train, y_train),
    val_data=(X_val, y_val),
    epochs=EPOCHS_PHASE2,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase2,
    class_weight=class_weight
)

print("\n✓ Phase 2 training completed!")

### 5. Combine Training Histories

# Combine histories
combined_history = {}
for key in history_phase1.keys():
    combined_history[key] = history_phase1[key] + history_phase2[key]

print(f"Combined history epochs: {len(combined_history['loss'])}")

### 6. Plot Training History

print("\nPlotting training history...\n")

plot_training_history(combined_history, figsize=(14, 5))
plt.savefig('../results/plots/mobilenet_training_history.png', dpi=300, bbox_inches='tight')
print("✓ Saved: mobilenet_training_history.png")

### 7. Evaluate on Test Set

print("\nEvaluating model on test set...\n")

# Reverse label encoding
reverse_encoding = {v: k for k, v in label_encoding.items()}
class_names = [reverse_encoding[i] for i in range(num_classes)]

# Evaluate
metrics = evaluate_model(model, X_test, y_test, class_names)

print("\n" + "="*70)
print("MODEL EVALUATION RESULTS")
print("="*70)
print(f"\nAccuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1_score']:.4f}")

### 8. Plot Confusion Matrix

print("\nPlotting confusion matrix...\n")

cm = np.array(metrics['confusion_matrix'])
plot_confusion_matrix(cm, class_names, figsize=(14, 12), normalize=False)
plt.savefig('../results/plots/mobilenet_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("✓ Saved: mobilenet_confusion_matrix.png")

# Normalized
plot_confusion_matrix(cm, class_names, figsize=(14, 12), normalize=True)
plt.savefig('../results/plots/mobilenet_confusion_matrix_normalized.png', dpi=300, bbox_inches='tight')
print("✓ Saved: mobilenet_confusion_matrix_normalized.png")

### 9. Plot Metrics Comparison

metrics_dict = {
    'Accuracy': metrics['accuracy'],
    'Precision': metrics['precision'],
    'Recall': metrics['recall'],
    'F1-Score': metrics['f1_score']
}

plot_metrics_comparison(metrics_dict, figsize=(10, 6))
plt.savefig('../results/plots/mobilenet_metrics.png', dpi=300, bbox_inches='tight')
print("✓ Saved: mobilenet_metrics.png")

### 10. Save Model

print("\nSaving model...\n")

model_path = save_model(model, MODEL_NAME, output_dir='../models/mobilenet')

print(f"\n✓ Model saved to: {model_path}")

### 11. Save Metrics and History

print("Saving metrics and history...\n")

# Save metrics
metrics_path = save_metrics(metrics, 'mobilenet_metrics.json', output_dir='../results/reports')

# Save history
with open('../results/reports/mobilenet_history.json', 'w') as f:
    history_serializable = {k: [float(v) for v in vals] for k, vals in combined_history.items()}
    json.dump(history_serializable, f, indent=4)

print("✓ Metrics saved")
print("✓ History saved")

### 12. Summary & Comparison

print("\n" + "="*70)
print("✓ MOBILENET V2 TRAINING COMPLETE")
print("="*70)

print(f"\n📊 Results Summary:")
print(f"  Model: MobileNetV2 (Transfer Learning)")
print(f"  Test Accuracy: {metrics['accuracy']:.4f}")
print(f"  Test Precision: {metrics['precision']:.4f}")
print(f"  Test Recall: {metrics['recall']:.4f}")
print(f"  Test F1-Score: {metrics['f1_score']:.4f}")

print(f"\n💾 Files Saved:")
print(f"  - Model: ../models/mobilenet/mobilenet_v2.h5")
print(f"  - Best Model: ../models/mobilenet/best_model.h5")
print(f"  - Metrics: ../results/reports/mobilenet_metrics.json")
print(f"  - History: ../results/reports/mobilenet_history.json")
print(f"  - Plots: ../results/plots/")

print(f"\n→ Next Step: 06_model_evaluation.ipynb")